# TL7 — Métodos basados en ejemplos

## Índice

1. [Introducción](#1-introducción)
2. [Clasificador por los K vecinos más próximos](#2-clasificador-por-los-k-vecinos-más-próximos)
3. [Regresión kernel](#3-regresión-kernel)
4. [Ejercicio con KNN](#4-ejercicio-con-knn)
5. [Ejercicio con regresión kernel](#5-ejercicio-con-regresión-kernel)

> **Cita:** Juan Ciscar, Alfons. *"Material docente de la asignatura Aprendizaje Automático (APA)"*. Grado de Inteligencia Artificial (GIAR), Escuela Técnica Superior de Ingeniería Informática (ETSINF), Dep. de Sistemas Informáticos y Computación (DSIC), Universitat Politècnica de València (UPV), primer semestre del curso 2026/27.

---
## 1. Introducción

**Clasificador *K nearest neighbor* (KNN):** retorna una etiqueta más votada entre los $K$ prototipos (datos) más cercanos.

**Regresión kernel:** Nadaraya-Watson con RBF Gaussiana

$$
\mathbb{E}[y \mid \boldsymbol{x}, \mathcal{D}] = \sum_{n=1}^{N} y_n\, w_n(\boldsymbol{x})
\quad \text{con} \quad
w_n(\boldsymbol{x}) = \mathcal{S}\left( \left\{ -\frac{1}{2h^2} \lVert \boldsymbol{x} - \boldsymbol{x}_{n'} \rVert^2 \right\}_{n'=1}^{N} \right)_n
$$

**Ingeniería de características:** la entrada $\boldsymbol{x}$ se sustituye por alguna transformación $\boldsymbol{\phi}(\boldsymbol{x})$.

**Objetivo:** aplicar KNN a tareas de clasificación y regresión kernel (sin y con ingeniería de características) a tareas de regresión.

---
## 2. Clasificador por los K vecinos más próximos

**KNN en scikit-learn:** `KNeighborsClassifier` con `n_neighbors=5` ($K$).

### 2.1 MNIST: lectura y reducción de la dimensión ($K = 400$)

Salida esperada: `(60000, 400) (60000,) (10000, 400) (10000,)`

In [ ]:
import numpy as np
import datasets
from sklearn.decomposition import PCA

ds = datasets.load_dataset("ylecun/mnist").with_format("numpy")
X_train = ds['train'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_train = ds['train'][:]['label'].astype(np.uint8)
X_test = ds['test'][:]['image'].astype(np.float32).reshape(-1, 28*28) / 255.
y_test = ds['test'][:]['label'].astype(np.uint8)

K = 400
pca = PCA(n_components=K)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

### 2.2 MNIST: precisión en test

Tras aprender `n_neighbors`.

Salida esperada:

```text
Precisión: 97.0% con 3.0000
Tiempo (hh:mm:ss): 00:00:11
```

In [ ]:
import time
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, ShuffleSplit, GridSearchCV

start = time.time()

clf = KNeighborsClassifier()
G = {"n_neighbors": [1, 3, 4, 5, 10]}
splitter = ShuffleSplit(n_splits=2, test_size=0.1, random_state=23)
GS = GridSearchCV(clf, G, scoring='accuracy', n_jobs=-1, cv=splitter, pre_dispatch=8)
acc = GS.fit(X_train, y_train).score(X_test, y_test)

print(f"Precisión: {acc:.1%} con {GS.best_params_['n_neighbors']:.4f}")
print('Tiempo (hh:mm:ss):', time.strftime('%H:%M:%S', time.gmtime(time.time() - start)))

---
## 3. Regresión kernel

**auto-mpg:** predicción de consumo en *miles per gallon*.

### 3.1 Carga y preproceso de auto-mpg

Mismo preproceso que en la práctica de regresión lineal: limpieza, binarización de `origin`, partición y estandarización.

In [ ]:
import numpy as np
import datasets
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

ds = datasets.load_dataset("scikit-learn/auto-mpg")
df = ds['train'].to_pandas()
df.drop(df[df.horsepower == "?"].index, inplace=True)
df = df.astype({'horsepower' : np.int64})
df['origin'] = df['origin'].map({1: 'USA', 2: 'Europe', 3: 'Japan'})
df = df.astype({'origin' : 'category'})
df.drop('car name', axis=1, inplace=True)
df = pd.concat([df, pd.get_dummies(df.origin, prefix='', prefix_sep='')], axis=1)
df.drop('origin', axis=1, inplace=True)
y = df.pop('mpg')
X = df

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### 3.2 Regresión kernel sin ingeniería de características

Para cada dato de test se calculan los pesos $w_n$ con un *softmax* de las distancias al cuadrado (restando el máximo por estabilidad numérica) y se predice la media ponderada de las $y_n$ de entrenamiento.

Salida esperada: `test R2 = 79.1%`

In [ ]:
from sklearn.metrics import r2_score

h = 1
negsqL2 = lambda x: -0.5 * x.T @ x / h

M = X_test.shape[0]
y_test_pred = np.zeros_like(y_test)
for i in np.arange(M):
    z = X_test[i] - X_train
    logits = np.apply_along_axis(negsqL2, 1, z)
    logits -= np.max(logits)
    w = np.exp(logits)
    w /= w.sum()
    y_test_pred[i] = y_train.T @ w

r2_test = r2_score(y_test, y_test_pred)
print(f'test R2 = {r2_test:.1%}')

### 3.3 Regresión kernel con ingeniería de características

Características polinómicas de grado 2.

Salida esperada: `test R2 = 88.1%`

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(2)
X_train = poly.fit_transform(X_train)
X_test = poly.transform(X_test)

for i in np.arange(M):
    z = X_test[i] - X_train
    logits = np.apply_along_axis(negsqL2, 1, z)
    logits -= np.max(logits)
    w = np.exp(logits)
    w /= w.sum()
    y_test_pred[i] = y_train.T @ w

r2_test = r2_score(y_test, y_test_pred)
print(f'test R2 = {r2_test:.1%}')

---
## 4. Ejercicio con KNN

**Ejercicio:** aplica KNN a Fashion-MNIST con $K^{(\text{pca})} = 600$ y a CIFAR-10 con $K^{(\text{pca})} = 1000$.

**Precisión (%) en test:** KNN no mejora los resultados de QLR.

|         | MNIST<br>$K=400$ | Fashion<br>$K=600$ | Fashion<br>$K=400$ | CIFAR-10<br>$K=1000$ | CIFAR-10<br>$K=400$ |
|---------|:----:|:----:|:----:|:----:|:----:|
| **LR**  | 92.6 | 84.4 | −    | 41.1 | −    |
| **LDA** | 87.3 | 81.6 | −    | 40.0 | −    |
| **QDA** | 95.8 | 76.6 | −    | 53.0 | −    |
| **QLR** | 98.3 | −    | 89.4 | −    | 52.6 |
| **KNN** | 97.0 | 85.8 | −    | 35.6 | −    |

### 4.1 Fashion-MNIST ($K = 600$)

#### 4.1.1 Lectura y reducción de la dimensión

Salida esperada: `(60000, 600) (60000,) (10000, 600) (10000,)`

In [ ]:
# Tu código aquí

#### 4.1.2 KNN

Salida esperada:

```text
Precisión: 85.8% con 4.0000
Tiempo (hh:mm:ss): 00:00:15
```

In [ ]:
# Tu código aquí

### 4.2 CIFAR-10 ($K = 1000$)

#### 4.2.1 Lectura y reducción de la dimensión

Salida esperada: `(50000, 1000) (50000,) (10000, 1000) (10000,)`

In [ ]:
# Tu código aquí

#### 4.2.2 KNN

Salida esperada:

```text
Precisión: 35.6% con 1.0000
Tiempo (hh:mm:ss): 00:00:18
```

In [ ]:
# Tu código aquí

---
## 5. Ejercicio con regresión kernel

**Ejercicio:** aplica regresión kernel a Fish y California housing, sin y con ingeniería de características.

**$R^2$ (%) en test:** similar a lineal en Fish, algo peor en housing.

|            | auto-mpg<br>sin | auto-mpg<br>con | Fish<br>sin | Fish<br>con | housing<br>sin | housing<br>con |
|------------|:----:|:----:|:----:|:----:|:----:|:----:|
| **Lineal** | 79.8 | 85.3 | 94.8 | 96.2 | 59.1 | 65.6 |
| **Kernel** | 79.1 | 88.1 | 91.9 | 96.4 | 42.7 | 60.0 |

### 5.1 Fish

#### 5.1.1 Carga, preproceso, partición y estandarización

In [ ]:
# Tu código aquí

#### 5.1.2 Regresión kernel sin ingeniería de características

Salida esperada: `test R2 = 91.9%`

In [ ]:
# Tu código aquí

#### 5.1.3 Regresión kernel con ingeniería de características

Salida esperada: `test R2 = 96.4%`

In [ ]:
# Tu código aquí

### 5.2 California housing

#### 5.2.1 Carga, partición y estandarización

In [ ]:
# Tu código aquí

#### 5.2.2 Regresión kernel sin ingeniería de características

Salida esperada: `test R2 = 42.7%`

In [ ]:
# Tu código aquí

#### 5.2.3 Regresión kernel con ingeniería de características

Salida esperada: `test R2 = 60.0%`

In [ ]:
# Tu código aquí